<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/nemotron-embed-finetuning/nemotron_3_embed_finetune_hf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning a Nemotron 3 text embedding model


In [1]:
import sys

if "google.colab" in sys.modules:
    !pip install -q -U bitsandbytes "torchao>=0.16.0"

In [2]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    bf16_supported = torch.cuda.is_bf16_supported()
elif torch.backends.mps.is_available():
    device = "mps"
    gpu_name = "Apple Silicon (MPS)"
    try:
        torch.zeros(1, device="mps", dtype=torch.bfloat16)
        bf16_supported = True
    except (RuntimeError, TypeError):
        bf16_supported = False
else:
    device = "cpu"
    gpu_name = "CPU ONLY"
    bf16_supported = False

print("Device:", device)
print("GPU:", gpu_name)
print("BF16 supported:", bf16_supported)

Device: mps
GPU: Apple Silicon (MPS)
BF16 supported: True


In [3]:
from datasets import load_dataset

print("Loading dataset...")
raw_ds = load_dataset("nvidia/Retrieval-Synthetic-NVDocs-v1",
                      split="train[:5000]")

sample_ds = raw_ds.shuffle(seed=42).select(range(min(3000, len(raw_ds))))

print(f"Sampled {len(sample_ds)} random examples.")


Loading dataset...
Sampled 3000 random examples.


In [4]:
print("Columns:", sample_ds.column_names)
print("Character length of first doc:", len(sample_ds[0]["text"]))
print("\n--- Raw Document Preview ---\n")
print(sample_ds[0]["text"][:600])

Columns: ['file_name', 'text', 'chunks', 'sections_structured', 'bundle_id', 'bundle_members', 'is_multi_doc', 'document_artifacts', 'qa_generation', 'deduplicated_qa_pairs', 'qa_evaluations']
Character length of first doc: 5132

--- Raw Document Preview ---

Whether tackling complex visualization challenges or creating Hollywood-caliber visual effects, artists and designers require powerful hardware to create their best work.
The latest application releases from Foundry, Chaos Group and Redshift by Maxon provide advanced features powered by NVIDIA RTX so creators can experience faster ray tracing and accelerated performance to elevate any design workflow.
Foundry Delivers New Features in Modo and Nuke
Foundry recently hosted Foundry LIVE, a series of virtual events where they announced the latest enhancements to their leading content creation appl


### Domain-Specific Retrieval Datasets Overview

| Dataset | Why it fits | Hugging Face URL |
| :--- | :--- | :--- |
| nvidia/Retrieval-Synthetic-NVDocs-v1 | Synthetic retrieval over NVIDIA documents. | https://huggingface.co/datasets/nvidia/Retrieval-Synthetic-NVDocs-v1 |
| Taylor658/synthetic-legal | Synthetic legal text dataset | https://huggingface.co/datasets/Taylor658/synthetic-legal |
| microsoft/mediflow | Large medical instruction dataset with retrieval/IR-adjacent examples and clinically grounded text. | https://huggingface.co/datasets/microsoft/mediflow |
| abhinand/MedEmbed-training-triplets-v1 | Synthetic triplets for medical retrieval training | https://huggingface.co/datasets/abhinand/MedEmbed-training-triplets-v1 |
| HFforLegal/case-law | Large legal corpus useful for retrieval indexing and RAG over legal decisions. | https://huggingface.co/datasets/HFforLegal/case-law |
| isaacus/legal-rag-bench | Retrieval-focused legal RAG dataset from a legal collection. | https://huggingface.co/datasets/isaacus/legal-rag-bench |
| isaacus/legal-rag-qa | Legal RAG QA dataset, good for retrieval-style benchmarking. | https://huggingface.co/datasets/isaacus/legal-rag-qa |
| isaacus/contractual-clause-retrieval | Explicit legal retrieval dataset centered on clause search. | https://huggingface.co/datasets/isaacus/contractual-clause-retrieval |
| PatronusAI/financebench | Finance-domain benchmark often used for retrieval over financial documents. | https://huggingface.co/datasets/PatronusAI/financebench |
| c3po-ai/edgar-corpus | SEC filings corpus, useful for long-document financial retrieval. | https://huggingface.co/datasets/c3po-ai/edgar-corpus |


In [5]:
import torch
import warnings
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
    GenerationConfig
)
import random
import numpy as np

warnings.filterwarnings("ignore")

qwen_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
llm_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {llm_id}...")
tokenizer = AutoTokenizer.from_pretrained(llm_id, clean_up_tokenization_spaces=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_id, quantization_config=qwen_bnb, device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
)

pipe.generation_config = GenerationConfig(
    max_new_tokens=50,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)


W1125 06:09:40.232000 81589 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading Qwen/Qwen2.5-1.5B-Instruct...


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
print(f"GPU memory allocated after loading Qwen: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU memory allocated after loading Qwen: 0.00 GB


In [7]:
def generate_synthetic_pair(example, chunk_length=1500):

    doc = str(example.get("text", "")).strip()
    if len(doc) > chunk_length:
        random_start = random.randint(0, len(doc) - chunk_length)
        start_pos = doc.find(". ", random_start)
        start_idx = start_pos + 2 if start_pos != -1 else random_start
        end_pos = doc.find(".", start_idx + chunk_length)
        end_idx = end_pos + 1 if end_pos != -1 else len(doc)
        chunk = doc[start_idx:end_idx].strip()
    else:
        chunk = doc

    prompt = f"<|im_start|>user\nRead the following document chunk and generate a single, short, specific question that is directly answered by it.\n\nDocument: {chunk}\n\nQuestion:<|im_end|>\n<|im_start|>assistant\n"

    outputs = pipe(
        prompt,
        truncation=True,
        return_full_text=False,
    )
    question = outputs[0]["generated_text"].strip()

    return {"anchor": question, "positive": chunk}


In [8]:
test_pair = generate_synthetic_pair(sample_ds[0])
print("GENERATED QUESTION (anchor):\n", test_pair["anchor"])
print("\nMATCHING CHUNK (positive):\n", test_pair["positive"][:400], "...")

GENERATED QUESTION (anchor):
 What exclusive discounts are offered to Foundry members in the US?

MATCHING CHUNK (positive):
 Learn more about how Foundry members in the U.S. can receive exclusive discounts and save on all Dell desktops, notebooks, servers, electronics and accessories.
Chaos Group to Release V-Ray 5 for Autodesk Maya
Chaos Group will soon release V-Ray 5 for Autodesk Maya, with a host of new GPU-accelerated features for lighting and materials.
Using LightMix in the new V-Ray Frame Buffer allows artists t ...


In [ ]:
print("Generating synthetic pairs (this may take a moment)...")
pair_ds = sample_ds.map(generate_synthetic_pair, remove_columns=sample_ds.column_names)
#pair_ds = sample_ds.select(range(10)).map(generate_synthetic_pair, remove_columns=sample_ds.column_names)

print("Generation Complete! Here are 3 examples:")
for i in range(min(3, len(pair_ds))):
    print(f"\n--- Example {i + 1} ---")
    print("Q:", pair_ds[i]["anchor"])
    print("A:", pair_ds[i]["positive"][:200], "...")

split_ds = pair_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split_ds["train"]
eval_ds = split_ds["test"]


Generating synthetic pairs (this may take a moment)...


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Generation Complete! Here are 3 examples:

--- Example 1 ---
Q: What products from Maxon are being showcased?
A: Learn more about how Foundry members in the U.S. can receive exclusive discounts and save on all Dell desktops, notebooks, servers, electronics and accessories.
Chaos Group to Release V-Ray 5 for Auto ...

--- Example 2 ---
Q: Which two technologies work together to provide an all-in-one solution?
A: Data scientists and DevOps teams can rely on solid support for the IBM Edge Application Manager no matter where on the network s edge their data is created and processed.
Running jobs on their own edg ...

--- Example 3 ---
Q: What orderings do these protocols provide?
A: We describe light-weight protocols for selective packet ordering in out-of-order networks that carry memory traffic. The protocols are designed for heterogeneous high-performance systems, in particula ...


In [12]:
print(f"Train pairs: {len(train_ds)} | Eval pairs: {len(eval_ds)}")

# Important for Colab: unload Qwen to free VRAM for Nemotron fine-tuning
import gc
del llm_model, pipe
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Train pairs: 8 | Eval pairs: 2
GPU memory after cleanup: 0.00 GB


In [ ]:
from transformers import AutoConfig
from sentence_transformers import SentenceTransformer
from peft import get_peft_model, LoraConfig, TaskType

embed_id = "nvidia/Nemotron-3-Embed-1B-BF16"

print(f"Loading base embedding model {embed_id}...")
embed_model = SentenceTransformer(
    embed_id, model_kwargs={"dtype": torch.bfloat16}, 
    )

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,)

peft_model = get_peft_model(embed_model[0].auto_model, lora_config)
embed_model[0].auto_model = peft_model
print("\nLoRA trainable parameters:")
peft_model.print_trainable_parameters()


Loading base embedding model nvidia/Nemotron-3-Embed-1B-BF16...


modules.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/36.3k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.28GB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
sample_vec = embed_model.encode("What is CUDA?")
print("Embedding shape:", sample_vec.shape)
print("First 8 numbers:", sample_vec[:8])

In [ ]:
from sentence_transformers.sentence_transformer.losses import (
    MultipleNegativesRankingLoss,)
from sentence_transformers.sentence_transformer.training_args import (
    SentenceTransformerTrainingArguments,)
from sentence_transformers.sentence_transformer.trainer import (
    SentenceTransformerTrainer,)

loss = MultipleNegativesRankingLoss(embed_model)

args = SentenceTransformerTrainingArguments(
    output_dir="./nemotron-embed-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",)

trainer = SentenceTransformerTrainer(
    model=embed_model, args=args, train_dataset=train_ds, loss=loss)

trainer.train()


In [ ]:
eval_queries = eval_ds["anchor"]
eval_docs = eval_ds["positive"]

def evaluate_retrieval(model, model_name):
    query_embs = model.encode(eval_queries)
    doc_embs = model.encode(eval_docs)
    similarities = np.dot(query_embs, doc_embs.T)
    top_doc_indices = np.argmax(similarities, axis=1)
    correct_indices = np.arange(len(eval_queries))
    correct_retrievals = np.sum(top_doc_indices == correct_indices)
    recall_at_1 = correct_retrievals / len(eval_queries)
    print(f"{model_name} Strict Recall@1: {recall_at_1 * 100:.2f}%")
    return recall_at_1

print("\n--- Evaluation Results ---")

original_model = SentenceTransformer(
    embed_id, model_kwargs={"dtype": torch.bfloat16})
finetuned_model = embed_model

original_score = evaluate_retrieval(original_model, "Original Model")
finetuned_score = evaluate_retrieval(finetuned_model, "Fine-Tuned Model")

improvement = finetuned_score - original_score
print(f"Absolute Improvement: +{improvement * 100:.2f}%")


In [ ]:
import numpy as np

orig_q = original_model.encode(eval_queries)
orig_d = original_model.encode(eval_docs)
orig_preds = np.argmax(np.dot(orig_q, orig_d.T), axis=1)

ft_q = finetuned_model.encode(eval_queries)
ft_d = finetuned_model.encode(eval_docs)
ft_preds = np.argmax(np.dot(ft_q, ft_d.T), axis=1)

# Find examples where Original failed but Fine-Tuned succeeded
wins = [i for i in range(len(eval_queries)) if orig_preds[i] != i and ft_preds[i] == i]
print(f"Found {len(wins)} queries fixed by fine-tuning!")

if wins:
    idx = wins[0]
    print("\nQUESTION:", eval_queries[idx])
    print("\n❌ Original model retrieved (WRONG):\n", eval_docs[orig_preds[idx]][:250], "...")
    print("\n✅ Fine-tuned model retrieved (RIGHT):\n", eval_docs[ft_preds[idx]][:250], "...")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(np.dot(orig_q[:15], orig_d[:15].T), cmap="viridis")
axes[0].set_title("Original Model (15x15 Similarity)")
axes[0].set_xlabel("Documents")
axes[0].set_ylabel("Queries")

axes[1].imshow(np.dot(ft_q[:15], ft_d[:15].T), cmap="viridis")
axes[1].set_title("Fine-Tuned Model (15x15 Similarity)")
axes[1].set_xlabel("Documents")
plt.tight_layout()
plt.show()